# Treinamento do detector de vacas (vista zenital) - GEDAI

**Antes de rodar**: menu *Ambiente de execucao -> Alterar tipo de ambiente de execucao -> GPU (T4)*.

Depois rode as celulas em ordem (Shift+Enter).

In [ ]:
# 1. Instala o Ultralytics (YOLO)
%pip -q install ultralytics

In [ ]:
# 2. Envie o arquivo dataset_yolo_v1.zip quando o botao aparecer
from google.colab import files
enviado = files.upload()  # selecione dataset_yolo_v1.zip

import zipfile, os
nome_zip = list(enviado.keys())[0]
# ZIPs criados no Windows usam '\' nos caminhos internos;
# convertemos para '/' antes de extrair
with zipfile.ZipFile(nome_zip) as z:
    for info in z.infolist():
        destino = info.filename.replace('\\', '/')
        if destino.endswith('/') or not destino:
            continue
        os.makedirs(os.path.dirname(destino) or '.', exist_ok=True)
        with z.open(info) as origem, open(destino, 'wb') as f:
            f.write(origem.read())
print('Extraido:', os.listdir('dataset_yolo'))

In [ ]:
# 3. Treina o YOLO11n a partir do modelo pre-treinado (fine-tuning)
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
resultado = model.train(
    data='dataset_yolo/data.yaml',
    epochs=60,
    imgsz=1280,      # resolucao das suas cameras; vacas ocupam boa parte do frame
    batch=8,
    patience=15,     # para cedo se a validacao parar de melhorar
    degrees=10, fliplr=0.5,  # augmentations uteis para vista de cima
)

In [ ]:
# 4. Avalia no conjunto de validacao e mostra exemplos
metricas = model.val()
print('mAP50:', metricas.box.map50)

from IPython.display import Image, display
import glob
for img in glob.glob('runs/detect/train/val_batch0_pred*.jpg')[:1]:
    display(Image(img))

In [ ]:
# 5. Baixa o modelo treinado para usar no seu PC
from google.colab import files
files.download('runs/detect/train/weights/best.pt')